# COMP 9130 — Final Hold-Out Evaluation & Cross-Experiment Comparison
## Notebook 04: Evaluation on 200-Image Hold-Out Test Set

**Author:** Sepehr Mansouri (Experiment 2 + Evaluation Lead)

**Purpose:** Evaluate all three experiments on the **same 200-image held-out
final test set** and produce cross-experiment comparison tables, per-subset
breakdowns, qualitative visualisations, and success criteria checks.

---

### What is the 200-image hold-out?

Before any model was trained, 200 images were permanently set aside and
**never seen during training or validation**:

| Subset | Count | Source | Purpose |
|---|---|---|---|
| ACD1K hold-out | 100 | Military camouflage (terrain-stratified) | Primary target domain |
| COD10K hold-out | 50 | Animal camouflage (super-class stratified) | Cross-domain generalisation |
| Noise distractors | 50 | Ordinary outdoor scenes (no targets) | False-positive rate (specificity) |

Every experiment's best checkpoint is evaluated on this identical set,
ensuring the comparison is fair.

### Success Criteria (Defined in Proposal)

On the 100-image ACD1K hold-out subset:
- **mIoU ≥ 0.65**
- **F1 (Dice) ≥ 0.75**

---
## 1. Environment Setup

In [ ]:
# Clone repo and install deps
import os
if not os.path.exists('/content/AI-final-project'):
    !git clone https://github.com/bing-er/AI-final-project.git
%cd /content/AI-final-project
!pip install -q transformers albumentations matplotlib

---
## 2. Dataset Download (if not already present)

In [ ]:
import os
from google.colab import files

# Kaggle credentials
if not os.path.exists('/root/.config/kaggle/kaggle.json'):
    files.upload()
    os.makedirs('/root/.config/kaggle', exist_ok=True)
    os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
    os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

# COD10K
if not os.path.exists('data/COD10K-v3/Train/Image'):
    !kaggle datasets download -d ivanomelchenkoim11/cod10k-dataset -p data/ --unzip
else:
    print("✅ COD10K exists")

# ACD1K
if not os.path.exists('data/dataset-splitM/Training/images'):
    !kaggle datasets download -d aalihhiader/military-camouflage-soldiers-dataset-mcs1k -p data/ --unzip
else:
    print("✅ ACD1K exists")

print("\n✅ All datasets ready")

---
## 3. Upload Checkpoints

Upload the best checkpoint from each experiment. You need:
- `exp2_best_model.pth` — from `outputs/exp2/final/best_model.pth`
- `exp3_best_model.pth` — from `outputs/exp3/final_lr6e5_50ep/best_model.pth`
- (Optional) `exp1_best_model.pth` — from Experiment 1 if available

**Skip this cell if checkpoints are already in the outputs/ folders.**

In [ ]:
# Upload checkpoints if not already present
import os

checkpoints = {
    'exp2': 'outputs/exp2/final/best_model.pth',
    'exp3': 'outputs/exp3/final_lr6e5_50ep/best_model.pth',
    # 'exp1': 'outputs/exp1/best_model.pth',  # uncomment when available
}

for name, path in checkpoints.items():
    if os.path.exists(path):
        print(f"✅ {name} checkpoint exists: {path}")
    else:
        print(f"⚠️  {name} checkpoint not found at {path}")
        print(f"    Upload it or adjust the path below.")

---
## 4. Core Evaluation Functions

In [ ]:
import sys
import json
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader
from transformers import SegformerForSemanticSegmentation
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

sys.path.insert(0, 'src')
from dataset import build_holdout_dataset, DATASET_STATS, get_val_transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


def load_segformer_checkpoint(checkpoint_path):
    """Load a SegFormer-B2 checkpoint saved by train_exp2/3.py."""
    model = SegformerForSemanticSegmentation.from_pretrained(
        'nvidia/segformer-b2-finetuned-ade-512-512',
        num_labels=1,
        ignore_mismatched_sizes=True,
    )
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    state_dict = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
    model.load_state_dict(state_dict, strict=True)
    model.to(device).eval()

    epoch = ckpt.get('epoch', '?')
    miou  = ckpt.get('val_mIoU', None)
    print(f"  Loaded: {checkpoint_path}")
    print(f"  Epoch: {epoch}, val mIoU: {miou:.4f}" if miou else f"  Epoch: {epoch}")
    return model


def predict_single(model, image_tensor):
    """Run inference on a single image tensor [1,3,H,W]. Returns prob map [H,W]."""
    with torch.no_grad():
        logits = model(pixel_values=image_tensor.to(device)).logits
        upsampled = F.interpolate(logits, size=(512, 512),
                                  mode='bilinear', align_corners=False)
        prob = torch.sigmoid(upsampled).squeeze().cpu().numpy()
    return prob


def compute_per_image_metrics(prob, mask_np):
    """Compute mIoU, F1, MAE for a single image."""
    pred_bin = (prob > 0.5).astype(np.float32)
    mask_bin = (mask_np > 0.5).astype(np.float32)

    # Foreground IoU
    inter = (pred_bin * mask_bin).sum()
    union = pred_bin.sum() + mask_bin.sum() - inter
    iou_fg = (inter + 1e-6) / (union + 1e-6)

    # Background IoU
    inter_bg = ((1 - pred_bin) * (1 - mask_bin)).sum()
    union_bg = (1 - pred_bin).sum() + (1 - mask_bin).sum() - inter_bg
    iou_bg = (inter_bg + 1e-6) / (union_bg + 1e-6)

    miou = (iou_fg + iou_bg) / 2
    f1 = (2 * inter + 1e-6) / (pred_bin.sum() + mask_bin.sum() + 1e-6)
    mae = np.abs(prob - mask_bin).mean()

    return {'mIoU': miou, 'F1': f1, 'MAE': mae, 'FP_pixels': pred_bin.sum()}


def evaluate_model_on_holdout(model, data_root='data/', splits_dir='splits/'):
    """Evaluate a model on all three hold-out subsets. Returns per-image results."""
    results = {}
    for subset_name in ['acd1k', 'cod10k', 'noise']:
        try:
            ds = build_holdout_dataset(data_root, subset_name, splits_dir=splits_dir)
        except FileNotFoundError as e:
            print(f"  ⚠️ {subset_name}: {e}")
            continue

        loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=2)
        subset_results = []

        for batch in loader:
            img = batch['image']        # [1, 3, 512, 512]
            mask = batch['mask']         # [1, 1, 512, 512]
            fname = batch['filename'][0]

            prob = predict_single(model, img)
            mask_np = mask.squeeze().numpy()
            m = compute_per_image_metrics(prob, mask_np)
            m['filename'] = fname
            m['subset'] = subset_name
            subset_results.append(m)

        results[subset_name] = subset_results
        avg_miou = np.mean([r['mIoU'] for r in subset_results])
        avg_f1   = np.mean([r['F1'] for r in subset_results])
        avg_mae  = np.mean([r['MAE'] for r in subset_results])
        print(f"  {subset_name.upper():>6s} ({len(subset_results):3d} imgs): "
              f"mIoU={avg_miou:.4f}  F1={avg_f1:.4f}  MAE={avg_mae:.4f}")

    return results


def denormalize(img_tensor, mean, std):
    """Undo normalization for display."""
    img = img_tensor.clone()
    for c in range(3):
        img[c] = img[c] * std[c] + mean[c]
    return img.clamp(0, 1).permute(1, 2, 0).numpy()


print("✅ Evaluation functions loaded")

---
## 5. Run Evaluation on All Experiments

In [ ]:
# Define experiment checkpoints — adjust paths as needed
experiment_configs = {
    # 'Exp1 (SINetV2)': 'outputs/exp1/best_model.pth',           # uncomment when available
    'Exp2 (Transfer)': 'outputs/exp2/final/best_model.pth',
    'Exp3 (Joint)':    'outputs/exp3/final_lr6e5_50ep/best_model.pth',
}

# Run evaluation
all_results = {}
models = {}

for exp_name, ckpt_path in experiment_configs.items():
    if not os.path.exists(ckpt_path):
        print(f"\n⚠️  {exp_name}: checkpoint not found at {ckpt_path}, skipping")
        continue

    print(f"\n{'='*60}")
    print(f"Evaluating: {exp_name}")
    print(f"{'='*60}")

    model = load_segformer_checkpoint(ckpt_path)
    models[exp_name] = model
    all_results[exp_name] = evaluate_model_on_holdout(model)

print(f"\n✅ Evaluation complete for {len(all_results)} experiments")

---
## 6. Cross-Experiment Comparison Table

In [ ]:
import pandas as pd

rows = []
for exp_name, subsets in all_results.items():
    for subset_name, per_img in subsets.items():
        mious = [r['mIoU'] for r in per_img]
        f1s   = [r['F1']   for r in per_img]
        maes  = [r['MAE']  for r in per_img]
        rows.append({
            'Experiment': exp_name,
            'Subset': subset_name.upper(),
            'N': len(per_img),
            'mIoU': f"{np.mean(mious):.4f}",
            'F1':   f"{np.mean(f1s):.4f}",
            'MAE':  f"{np.mean(maes):.4f}",
        })

    # Combined camouflage (ACD1K + COD10K)
    camo = subsets.get('acd1k', []) + subsets.get('cod10k', [])
    if camo:
        rows.append({
            'Experiment': exp_name,
            'Subset': 'ALL CAMO',
            'N': len(camo),
            'mIoU': f"{np.mean([r['mIoU'] for r in camo]):.4f}",
            'F1':   f"{np.mean([r['F1'] for r in camo]):.4f}",
            'MAE':  f"{np.mean([r['MAE'] for r in camo]):.4f}",
        })

df = pd.DataFrame(rows)
print("\n" + "="*70)
print("CROSS-EXPERIMENT COMPARISON — 200-IMAGE HOLD-OUT")
print("="*70)
print(df.to_string(index=False))
print("="*70)

---
## 7. Success Criteria Check (ACD1K Hold-Out)

In [ ]:
print("\n" + "="*60)
print("SUCCESS CRITERIA CHECK — ACD1K Hold-Out (100 images)")
print("  Required: mIoU >= 0.65  AND  F1 >= 0.75")
print("="*60)

for exp_name, subsets in all_results.items():
    acd1k = subsets.get('acd1k', [])
    if not acd1k:
        continue
    miou = np.mean([r['mIoU'] for r in acd1k])
    f1   = np.mean([r['F1']   for r in acd1k])
    miou_pass = '✅ PASS' if miou >= 0.65 else '❌ FAIL'
    f1_pass   = '✅ PASS' if f1   >= 0.75 else '❌ FAIL'
    print(f"\n  {exp_name}:")
    print(f"    mIoU = {miou:.4f}  {miou_pass}")
    print(f"    F1   = {f1:.4f}  {f1_pass}")

---
## 8. False-Positive Rate on Noise Images

In [ ]:
print("\nFALSE-POSITIVE RATE — Noise Distractors (50 images)")
print("  (proportion of noise images where model predicts >1% foreground)\n")

FP_THRESHOLD = 0.01  # 1% of pixels

for exp_name, subsets in all_results.items():
    noise = subsets.get('noise', [])
    if not noise:
        continue
    total_pixels = 512 * 512
    fp_count = sum(1 for r in noise
                   if r['FP_pixels'] > FP_THRESHOLD * total_pixels)
    fpr = fp_count / len(noise)
    avg_mae = np.mean([r['MAE'] for r in noise])
    print(f"  {exp_name}:")
    print(f"    FPR = {fp_count}/{len(noise)} = {fpr:.2%}")
    print(f"    Avg MAE on noise = {avg_mae:.4f}")

---
## 9. Qualitative Visualisations — Prediction Overlays

For each experiment, show a grid of sample images with:
- **Left:** Original image
- **Centre:** Ground-truth mask
- **Right:** Predicted mask overlay (green = correct, red = false positive, blue = missed)

In [ ]:
def visualise_predictions(model, dataset, exp_name, num_samples=6,
                          stats_key='ACD1K'):
    """Show prediction overlays for sample images from a hold-out subset."""
    stats = DATASET_STATS[stats_key]
    indices = np.random.choice(len(dataset), min(num_samples, len(dataset)),
                               replace=False)

    fig, axes = plt.subplots(len(indices), 4, figsize=(20, 4*len(indices)))
    if len(indices) == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(f'{exp_name} — Prediction Visualisation', fontsize=16, y=1.01)

    for row, idx in enumerate(indices):
        batch = dataset[idx]
        img_tensor = batch['image'].unsqueeze(0)
        mask_np = batch['mask'].squeeze().numpy()
        fname = batch['filename']

        # Predict
        prob = predict_single(model, img_tensor)
        pred_bin = (prob > 0.5).astype(np.float32)

        # Denormalize for display
        img_display = denormalize(batch['image'], stats['mean'], stats['std'])

        # Build overlay: green=TP, red=FP, blue=FN
        overlay = img_display.copy()
        tp = (pred_bin > 0.5) & (mask_np > 0.5)
        fp = (pred_bin > 0.5) & (mask_np < 0.5)
        fn = (pred_bin < 0.5) & (mask_np > 0.5)
        overlay[tp] = overlay[tp] * 0.5 + np.array([0, 1, 0]) * 0.5   # green
        overlay[fp] = overlay[fp] * 0.5 + np.array([1, 0, 0]) * 0.5   # red
        overlay[fn] = overlay[fn] * 0.5 + np.array([0, 0, 1]) * 0.5   # blue

        # Compute per-image metrics
        m = compute_per_image_metrics(prob, mask_np)

        # Plot
        axes[row, 0].imshow(img_display)
        axes[row, 0].set_title(fname, fontsize=9)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(mask_np, cmap='gray')
        axes[row, 1].set_title('Ground Truth')
        axes[row, 1].axis('off')

        axes[row, 2].imshow(prob, cmap='hot', vmin=0, vmax=1)
        axes[row, 2].set_title(f'Prediction (prob)')
        axes[row, 2].axis('off')

        axes[row, 3].imshow(overlay)
        axes[row, 3].set_title(f'Overlay  mIoU={m["mIoU"]:.3f} F1={m["F1"]:.3f}')
        axes[row, 3].axis('off')

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='green', label='True Positive'),
                       Patch(facecolor='red',   label='False Positive'),
                       Patch(facecolor='blue',  label='False Negative (missed)')]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3,
               fontsize=11, bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout()
    plt.savefig(f'outputs/eval_{exp_name.replace(" ", "_").replace("(","").replace(")","")}_vis.png',
                dpi=150, bbox_inches='tight')
    plt.show()

print("✅ Visualisation function ready")

In [ ]:
# Visualise ACD1K predictions for each experiment
np.random.seed(42)  # reproducible sample selection

for exp_name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Visualising: {exp_name} on ACD1K hold-out")
    print(f"{'='*50}")
    ds = build_holdout_dataset('data/', 'acd1k', splits_dir='splits/')
    visualise_predictions(model, ds, exp_name, num_samples=6,
                          stats_key='ACD1K')

In [ ]:
# Visualise COD10K predictions (animal camouflage transfer)
np.random.seed(42)

for exp_name, model in models.items():
    print(f"\nVisualising: {exp_name} on COD10K hold-out")
    ds = build_holdout_dataset('data/', 'cod10k', splits_dir='splits/')
    visualise_predictions(model, ds, exp_name, num_samples=4,
                          stats_key='COD10K')

In [ ]:
# Visualise noise predictions (should show almost no detection)
np.random.seed(42)

for exp_name, model in models.items():
    print(f"\nVisualising: {exp_name} on Noise distractors")
    ds = build_holdout_dataset('data/', 'noise', splits_dir='splits/')
    visualise_predictions(model, ds, exp_name, num_samples=4,
                          stats_key='COD10K')

---
## 10. Side-by-Side Comparison — Same Images Across Experiments

Shows the same ACD1K image predicted by each experiment side-by-side.

In [ ]:
def side_by_side_comparison(models_dict, data_root='data/',
                           splits_dir='splits/', num_images=5):
    """Show same images across all experiments for direct comparison."""
    ds = build_holdout_dataset(data_root, 'acd1k', splits_dir=splits_dir)
    stats = DATASET_STATS['ACD1K']
    exp_names = list(models_dict.keys())
    n_exps = len(exp_names)

    np.random.seed(42)
    indices = np.random.choice(len(ds), min(num_images, len(ds)), replace=False)

    fig, axes = plt.subplots(len(indices), n_exps + 2,
                              figsize=(4*(n_exps+2), 4*len(indices)))
    if len(indices) == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle('Cross-Experiment Comparison — Same ACD1K Images',
                 fontsize=16, y=1.01)

    for row, idx in enumerate(indices):
        batch = ds[idx]
        img_tensor = batch['image'].unsqueeze(0)
        mask_np = batch['mask'].squeeze().numpy()
        img_display = denormalize(batch['image'], stats['mean'], stats['std'])

        # Column 0: Original image
        axes[row, 0].imshow(img_display)
        axes[row, 0].set_title(batch['filename'], fontsize=9)
        axes[row, 0].axis('off')

        # Column 1: Ground truth
        axes[row, 1].imshow(mask_np, cmap='gray')
        axes[row, 1].set_title('Ground Truth')
        axes[row, 1].axis('off')

        # Columns 2+: Each experiment's prediction
        for col, exp_name in enumerate(exp_names):
            model = models_dict[exp_name]
            prob = predict_single(model, img_tensor)
            pred_bin = (prob > 0.5).astype(np.float32)
            m = compute_per_image_metrics(prob, mask_np)

            # Overlay
            overlay = img_display.copy()
            tp = (pred_bin > 0.5) & (mask_np > 0.5)
            fp = (pred_bin > 0.5) & (mask_np < 0.5)
            fn = (pred_bin < 0.5) & (mask_np > 0.5)
            overlay[tp] = overlay[tp] * 0.5 + np.array([0, 1, 0]) * 0.5
            overlay[fp] = overlay[fp] * 0.5 + np.array([1, 0, 0]) * 0.5
            overlay[fn] = overlay[fn] * 0.5 + np.array([0, 0, 1]) * 0.5

            axes[row, col+2].imshow(overlay)
            axes[row, col+2].set_title(
                f'{exp_name}\nmIoU={m["mIoU"]:.3f}', fontsize=10)
            axes[row, col+2].axis('off')

    plt.tight_layout()
    plt.savefig('outputs/eval_cross_experiment_comparison.png',
                dpi=150, bbox_inches='tight')
    plt.show()

side_by_side_comparison(models, num_images=6)

---
## 11. Training Curves — Loss & mIoU Over Epochs

In [ ]:
def plot_training_curves(history_paths, title_prefix=''):
    """Plot training/validation loss and mIoU curves from history.json files."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for label, path in history_paths.items():
        if not os.path.exists(path):
            print(f"  ⚠️ {label}: {path} not found, skipping")
            continue
        with open(path) as f:
            h = json.load(f)
        epochs = [r['epoch'] for r in h]
        ax1.plot(epochs, [r['val_loss'] for r in h], label=f'{label} (val)')
        ax1.plot(epochs, [r['train_loss'] for r in h], '--', alpha=0.4,
                 label=f'{label} (train)')
        ax2.plot(epochs, [r['val_mIoU'] for r in h], label=f'{label} (val)')
        ax2.plot(epochs, [r['train_mIoU'] for r in h], '--', alpha=0.4,
                 label=f'{label} (train)')

    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title(f'{title_prefix}Training & Validation Loss')
    ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

    ax2.set_xlabel('Epoch'); ax2.set_ylabel('mIoU')
    ax2.set_title(f'{title_prefix}Training & Validation mIoU')
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'outputs/eval_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


# Plot training curves for all experiments
history_paths = {
    # 'Exp1 (SINetV2)':        'outputs/exp1/history.json',  # uncomment when available
    'Exp2 Stage1 (COD10K)':  'outputs/exp2/stage1/history.json',
    'Exp2 Stage2 (ACD1K)':   'outputs/exp2/final/history.json',
    'Exp3 (Joint)':          'outputs/exp3/final_lr6e5_50ep/history.json',
}

plot_training_curves(history_paths)

---
## 12. Save All Results to JSON

In [ ]:
# Save comprehensive results
output = {
    'experiments': {},
}

for exp_name, subsets in all_results.items():
    exp_out = {}
    for subset_name, per_img in subsets.items():
        mious = [r['mIoU'] for r in per_img]
        f1s   = [r['F1']   for r in per_img]
        maes  = [r['MAE']  for r in per_img]
        exp_out[subset_name] = {
            'n': len(per_img),
            'mIoU_mean': round(float(np.mean(mious)), 4),
            'mIoU_std':  round(float(np.std(mious)), 4),
            'F1_mean':   round(float(np.mean(f1s)), 4),
            'F1_std':    round(float(np.std(f1s)), 4),
            'MAE_mean':  round(float(np.mean(maes)), 4),
            'MAE_std':   round(float(np.std(maes)), 4),
        }
    output['experiments'][exp_name] = exp_out

os.makedirs('outputs/eval', exist_ok=True)
with open('outputs/eval/final_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("✅ Results saved → outputs/eval/final_results.json")
print(json.dumps(output, indent=2))

---
## 13. Push Results to GitHub

In [ ]:
!git config --global user.email "smansouri7@my.bcit.ca"
!git config --global user.name "sepehr-mansouri"
!git add outputs/eval/
!git commit -m "Final hold-out evaluation: cross-experiment comparison + visualisations"
!git push

# Download key outputs
from google.colab import files
files.download('outputs/eval/final_results.json')